# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided, reproducible workflow for loading and exploring a dataset using the `mlcroissant` library, referencing all entities by their Croissant `@id` fields for precision.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed. Un-comment to install locally.
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# WARNING: Do not access dataset.metadata as a dict – it's a Croissant Metadata object
meta = dataset.metadata
print(f"Dataset title: {meta.name}")
print(f"Description: {meta.description}\n")

## 2. Data Overview
Review available record sets, their `@id`s, and fields in the dataset. This will help us identify which `@id`s to use for loading and manipulating data.

In [ ]:
# List all record sets with their @id and fields

print("Available Record Sets (@id):\n")
for record_set in meta.record_sets:
    print(f"  - {record_set.id} (name: {record_set.name})")
    if hasattr(record_set, 'fields'):
        
        print("    Fields/columns under this record set:")
        for field in record_set.fields:
            desc = field.description if hasattr(field, 'description') and field.description is not None else ''
            print(f"      - {field.id} (name: {field.name}, type: {getattr(field, 'data_type', 'N/A')}) {desc}")
    print("")

## 3. Data Extraction
Load data from record sets into pandas DataFrames for analysis, referencing by `@id` as obtained above.

In [ ]:
# ---
# Replace these with the actual record set @id(s) you want to extract, as discovered above:

record_set_ids = [rs.id for rs in meta.record_sets]

# Load all tables into a dictionary of DataFrames
dataframes = {}
for record_set_id in record_set_ids:
    try:
        # Each record will be a dict with field @id as key
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {record_set_id}: {df.shape[0]} rows, {df.shape[1]} columns")
    except Exception as e:
        print(f"Could not extract records for {record_set_id}: {e}")

# For illustration, select the first loaded record set (if any)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns for {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Here, we use one of the numeric columns (for example 'http://mlcommons.org/croissant/abundance') as found by its `@id`. If the precise `@id` is not clear, refer to the list in Section 2.

In [ ]:
###-- Replace these with the correct @id as found above --###

# Choose the record set for EDA
record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Guess likely numeric fields to analyze
def guess_numeric_fields(df):
    for col in df.columns:
        try:
            series = pd.to_numeric(df[col])
            if not series.isnull().all():
                yield col
        except Exception:
            continue

numeric_fields = list(guess_numeric_fields(df))
print(f"Candidate numeric fields (@id): {numeric_fields}\n")

# Pick a numeric field @id, e.g., abundance, number of peptides, or molecular weight
if numeric_fields:
    numeric_field_id = numeric_fields[0]  # the first numeric field found
else:
    raise ValueError("No numeric fields found in the table.")

# Threshold for filtering
threshold = df[numeric_field_id].astype(float).mean()  # Use mean as an example threshold

filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold].copy()

print(f"Filtered records with {numeric_field_id} > {threshold:.3f} (using @id):")
display(filtered_df.head())

# Normalization (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by another field – any string/categorical field
string_fields = [col for col in df.columns if df[col].dtype == object and df[col].nunique() < 100]
group_field_id = None
if string_fields:
    group_field_id = string_fields[0]
    grouped_df = (
        filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    )
    print(f"Grouped statistics by: {group_field_id} (@id):")
    display(grouped_df.head())
else:
    print("No suitable categorical field found for group-by analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Example: histogram of the selected numeric field, or a boxplot of values grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id} (@id)')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If group_field_id chosen, show boxplot
if group_field_id is not None:
    plt.figure(figsize=(12,4))
    sns.boxplot(
        x=filtered_df[group_field_id],
        y=pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    )
    plt.title(f'{numeric_field_id} grouped by {group_field_id} (@id)')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to:
- Load Croissant-format datasets with `mlcroissant`
- Enumerate available record sets and fields by their `@id`
- Extract data from any record set for analysis
- Perform filtering, normalization, and grouping on numeric data fields (referenced by `@id`)
- Visualize distributions using Matplotlib/Seaborn

All operations referenced dataset fields strictly by their Croissant `@id` for reproducibility and clarity. Continue your workflow by adapting the sections above for deeper biological analysis or integration with domain-specific tools.